<a href="https://colab.research.google.com/github/chetnapriyadarshini/Llama_Index_Query_Engine/blob/main/llama_QA_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install llama-index

In [7]:
from llama_index.core.llms import ChatMessage
from llama_index.llms.openai import OpenAI
import os
import openai

In [8]:
messages = [
         ChatMessage(role="system", content="You are an AI assistant to the user."),
         ChatMessage(role="user", content="What is the revenue of Uber in 2021?"),
     ]

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
with open('/content/drive/MyDrive/GenAI/openAI_apiKey.txt', 'r') as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

In [ ]:
!pip install llama-index-readers-file

In [6]:
resp = OpenAI(api_key=os.environ["OPENAI_API_KEY"]).chat(messages)
print(resp)

NameError: name 'OpenAI' is not defined

In [4]:
import nest_asyncio
nest_asyncio.apply()

In [5]:
from pathlib import Path
from llama_index.readers.file import PDFReader

loader = PDFReader()
documents = loader.load_data(file=Path('/content/drive/MyDrive/GenAI/Llama-llama-where-is-your-payjama/NYSE_UBER_2024.pdf'))

In [6]:
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core import VectorStoreIndex
from IPython.display import display, HTML

# create parser and parse document into nodes
parser = SimpleNodeParser.from_defaults()
nodes = parser.get_nodes_from_documents(documents)
# build the index
index = VectorStoreIndex(nodes)

In [15]:
# Construct Query Engine
query_engine = index.as_query_engine()

# Query the engine.
response = query_engine.query("What is the revenue of Uber in 2024?")

# print the synthesized response.
display(HTML(f'<p style="font-size:20px">{response.response}</p>'))

In [17]:
from llama_index.core.readers import SimpleDirectoryReader

# Create a new document reader for a text file.
reader = SimpleDirectoryReader(input_files=["/content/drive/MyDrive/GenAI/Llama-llama-where-is-your-payjama/paul_graham_essay.txt"])

# Load data from the new document.
documents = reader.load_data()

# Create new nodes from the new documents.
new_nodes = parser.get_nodes_from_documents(documents)

# Insert new nodes into the existing index.
index.insert_nodes(new_nodes)

# Reconstruct the query engine.
query_engine = index.as_query_engine()

# Query the engine with a new question.
response = query_engine.query("Why did Paul Graham start YC.")

# Print the synthesized response.
display(HTML(f'<p style="font-size:20px">{response.response}</p>'))

In [19]:
from llama_index.core import SummaryIndex

summary_index = SummaryIndex(new_nodes)
vector_index = VectorStoreIndex(new_nodes)

In [20]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)
vector_query_engine = vector_index.as_query_engine()

In [21]:
from llama_index.core.tools.query_engine import QueryEngineTool

list_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description="Useful for summarisation questions related to Paul Graham eassy on What I Worked On.",
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description="Useful for retrieving specific context from Paul Graham essay on What I Worked On.",
)

In [22]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors.llm_selectors import LLMSingleSelector

query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        list_tool,
        vector_tool,
    ],
)

In [3]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.readers import SimpleDirectoryReader

# Load Documents
lyft_docs = SimpleDirectoryReader(input_files=["/content/drive/MyDrive/GenAI/Llama-llama-where-is-your-payjama/Lyft-Annual-Report-2021.pdf"]).load_data()
uber_docs = SimpleDirectoryReader(input_files=["/content/drive/MyDrive/GenAI/Llama-llama-where-is-your-payjama/uber_2021.pdf"]).load_data()

In [10]:
# Create Indicies
from llama_index.core import VectorStoreIndex
lyft_index = VectorStoreIndex.from_documents(lyft_docs)
uber_index = VectorStoreIndex.from_documents(uber_docs)

In [11]:
lyft_engine = lyft_index.as_query_engine(similarity_top_k=3)

uber_engine = uber_index.as_query_engine(similarity_top_k=3)

In [12]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

query_engine_tools = [
    QueryEngineTool(
        query_engine=lyft_engine,
        metadata=ToolMetadata(name='lyft_10k', description='Provides information about Lyft financials for year 2021')
    ),
    QueryEngineTool(
        query_engine=uber_engine,
        metadata=ToolMetadata(name='uber_10k', description='Provides information about Uber financials for year 2021')
    ),
]

In [ ]:
!pip install llama-index-question-gen-openai

In [13]:
from llama_index.core.query_engine import SubQuestionQueryEngine
s_engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=query_engine_tools)

In [14]:
from IPython.display import display, HTML
response = await s_engine.aquery('Compare revenue growth of Uber and Lyft from 2020 to 2021')
display(HTML(f'<p style="font-size:20px">{response.response}</p>'))

Generated 4 sub questions.
[uber_10k] Q: What was the revenue of Uber in 2020?
[uber_10k] Q: What was the revenue of Uber in 2021?
[lyft_10k] Q: What was the revenue of Lyft in 2020?
[lyft_10k] Q: What was the revenue of Lyft in 2021?
[uber_10k] A: The revenue of Uber in 2021 was not explicitly mentioned in the provided context information.
[uber_10k] A: The revenue of Uber in 2020 was $11.1 billion.
[lyft_10k] A: Lyft's revenue in 2021 was $3,208,323.
[lyft_10k] A: The revenue of Lyft in 2020 was $2,364,681,000.


### Using Google Gemini Models with LlamaIndex

To use Google's Gemini models, you'll need a Google API key. If you don't have one, you can create one in Google AI Studio (<a href="https://aistudio.google.com/app/apikey" target="_blank">aistudio.google.com/app/apikey</a>).

Once you have your API key, add it to Colab's secrets manager (the '🔑' icon in the left panel) and name it `GOOGLE_API_KEY`. Then, we can securely access it and configure the Gemini LLM for LlamaIndex.


In [18]:
# Install the necessary package for Gemini LLM integration
#!pip install llama-index-llms-gemini google-generativeai

# Import the required libraries
from llama_index.core import VectorStoreIndex, Settings
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.openai import OpenAIEmbedding
import os
from google.colab import userdata

# Get your Google API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GeminiApiKey')

# Initialize the Gemini LLM
llm = Gemini(api_key=GOOGLE_API_KEY)

# Keep the OpenAI embedding model as it is working and uses your OpenAI API key
embed_model = OpenAIEmbedding(model="text-embedding-ada-002",
                                api_key=os.environ["OPENAI_API_KEY"])

# Configure global settings using llama_index.core.Settings
Settings.llm = llm
Settings.embed_model = embed_model

# Re-create the index and query engine
index = VectorStoreIndex.from_documents(lyft_docs + uber_docs)
query_engine = index.as_query_engine()



/tmp/ipykernel_20453/1336452671.py:15: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  llm = Gemini(api_key=GOOGLE_API_KEY)


In [ ]:
# Query the engine with the desired question
response = query_engine.query("What was uber's revenue in 2024?")
print(response)